# 03 - A network twice as deep as big (P2, lowest priority)
Protocol: `results/deep_network_2026-11/PROTOCOL.md`. Training writes a checkpoint per epoch to Drive.

In [ ]:
# 0.1 Setup: Google Drive, the code at the pinned commit, pinned packages (~3-5 min)
import json, os, subprocess, sys
COMMIT = "91ebd88a06d6eee73a1656dba7f122c84a7664a4"          # code under test (branch checkpoint)
BRANCH = "checkpoint"
REPO_URL = "https://github.com/dimitrijepesic/Pasado.git"   # default source of the code
DRIVE_ROOT = "/content/drive/MyDrive/pasado_checkpoint_2026_11"
BUNDLE = DRIVE_ROOT + "/checkpoint.bundle"   # optional: used instead of GitHub if present
LOCAL_TEST = os.environ.get("PASADO_NB_LOCAL_TEST", "0") == "1"
PIN_TORCH = ["torch==2.11.0", "torchvision==0.26.0"]        # the stack of the October A100 runs
TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
PINS = ["numpy==2.5.1", "scipy==1.18.0", "scikit-learn==1.9.0", "mpmath==1.4.1", "affapy==0.1"]
if LOCAL_TEST:
    REPO, DRIVE_ROOT = os.environ["PASADO_NB_REPO"], os.environ["PASADO_NB_OUT"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    REPO = "/content/Pasado"
    if not os.path.isdir(REPO + "/.git"):
        src = BUNDLE if os.path.exists(BUNDLE) else REPO_URL
        extra = [] if src == BUNDLE else ["--branch", BRANCH]
        subprocess.run(["git", "clone", *extra, src, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", COMMIT], check=True)
    import importlib.metadata as md
    if md.version("torch").split("+")[0] != PIN_TORCH[0].split("==")[1]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIN_TORCH,
                        "--index-url", TORCH_INDEX], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
print("code at", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                                text=True).stdout.strip())

In [ ]:
# 0.2 Environment record and result folder on Drive (one folder per GPU model)
sys.path.insert(0, os.path.join(REPO, "colab"))
import nbutil
GPU = nbutil.gpu_tag()
OUT = os.path.join(DRIVE_ROOT, "03_deep_network", GPU)
CACHE = os.path.join(DRIVE_ROOT, "inductor_cache", GPU)      # compilation is paid once per GPU model
os.makedirs(CACHE, exist_ok=True)
R = nbutil.Runner(REPO, OUT, extra_env={"TORCHINDUCTOR_CACHE_DIR": CACHE})
T = nbutil.LOCAL_TEST
DEV = "cpu" if T else "cuda"
NETS = ["3layer"] if T else ["3layer", "4layer", "5layer", "big"]
if T:
    BIG_CHUNK = 8
else:
    free_mib = int(nbutil.sh("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits").split()[0])
    BIG_CHUNK = 64 if free_mib >= 70 * 1024 else 32
print(json.dumps(R.env_record, indent=1))
print("results ->", OUT, "| big chunk", BIG_CHUNK)
if not T:
    print(nbutil.sh("nvidia-smi"))

In [ ]:
# Train deep8 (~10-15 min on an A100; resumes per epoch), then the bounds of big and deep8 (~10 min)
deep_dir = R.path("deep8", "x")[:-2]
R.run("A_train_deep8", ["experiments/train_deep_network.py", "--out-dir", deep_dir, "--device", DEV]
      + (["--epochs", 1, "--limit-batches", 3] if T else []), estimate_min=15)
print(open(os.path.join(deep_dir, "training.json")).read()[:300])
R.run("B_bounds_deep8", ["experiments/measure_deep_network.py", "--deep-dir", deep_dir, "--device", DEV,
                         "--out", R.path("bounds.csv")]
      + (["--num-images", 1, "--eps-indices", 0, "--splits", 1, "--smoke"] if T else []), estimate_min=10)